# 6주차 세분화(Segmentation) 실습 — U-Net vs SegFormer vs SAM
**Oxford-IIIT Pet**(반려동물 전경/배경 분할)에서 다음 세 가지 방식을 비교합니다.

| 방식 | 모델 | 특징 |
|---|---|---|
| 클래식 전용 학습 | **U-Net** (ResNet34 인코더, ImageNet 사전학습) | 작업용 데이터로 파인튜닝 |
| Transformer 전용 학습 | **SegFormer-B0** (`nvidia/mit-b0`, Hugging Face) | 경량 Transformer, 파인튜닝 |
| 파운데이션 모델 | **SAM** (`facebook/sam-vit-base`, Hugging Face) | 재학습 없이 박스/점 프롬프트로 zero-shot 분할 |

평가 지표: **IoU, Dice** (픽셀 단위). 실행: `런타임 > 런타임 유형 변경 > T4 GPU` 후 `런타임 > 모두 실행` (약 15분). 마지막 셀이 `results_w6.zip`을 다운로드하고 결과 JSON을 출력합니다.

## 1. 환경 설정

In [ ]:
!pip -q install segmentation-models-pytorch
import os, json, time, random, zipfile
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import torchvision, transformers
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import distance_transform_edt
import segmentation_models_pytorch as smp
from transformers import SegformerForSemanticSegmentation, SamModel, SamProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
ENV = {"device": device, "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "CPU",
       "torch": torch.__version__, "transformers": transformers.__version__, "smp": smp.__version__}
print(ENV)
SEED = 42
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
os.makedirs("results_w6", exist_ok=True)

# 실험 설정 (Colab T4 기준으로 가볍게)
IMG = 256            # 학습/평가 해상도
N_TRAIN, N_VAL, N_TEST = 1000, 200, 300
EPOCHS, BATCH = 8, 16

## 2. 데이터: Oxford-IIIT Pet (trimap → 전경/배경 이진 마스크)

In [ ]:
# torchvision 내장 데이터셋 (images ~760MB + annotations ~19MB)
trainval = torchvision.datasets.OxfordIIITPet("data", split="trainval", target_types="segmentation", download=True)
testset  = torchvision.datasets.OxfordIIITPet("data", split="test",     target_types="segmentation", download=True)

def to_binary(trimap, strict=False):
    t = np.array(trimap)
    if strict: return (t == 1).astype(np.uint8)       # 경계(3) 제외한 '확실한' 전경만
    return ((t == 1) | (t == 3)).astype(np.uint8)   # 1=pet, 3=경계 → 전경 / 2=배경

rng = np.random.RandomState(SEED)
tv_idx = rng.permutation(len(trainval)); te_idx = rng.permutation(len(testset))[:N_TEST]
tr_idx, va_idx = tv_idx[:N_TRAIN], tv_idx[N_TRAIN:N_TRAIN + N_VAL]

def load_resized(ds, idxs):
    X = np.zeros((len(idxs), IMG, IMG, 3), np.uint8); Y = np.zeros((len(idxs), IMG, IMG), np.uint8); Ys = Y.copy()
    for k, i in enumerate(idxs):
        img, tri = ds[int(i)]
        X[k] = np.array(img.convert("RGB").resize((IMG, IMG), Image.BILINEAR))
        tri = tri.resize((IMG, IMG), Image.NEAREST); Y[k] = to_binary(tri); Ys[k] = to_binary(tri, strict=True)
    return X, Y, Ys
t0 = time.time()
Xtr, Ytr, _ = load_resized(trainval, tr_idx); Xva, Yva, _ = load_resized(trainval, va_idx); Xte, Yte, Yte_strict = load_resized(testset, te_idx)
# SAM은 원본 해상도 이미지를 사용
test_orig = [(testset[int(i)][0].convert("RGB"), to_binary(testset[int(i)][1])) for i in te_idx]
print(f"loaded in {time.time()-t0:.1f}s", Xtr.shape, Xva.shape, Xte.shape)

DATA = {"name": "Oxford-IIIT Pet (Parkhi et al., 2012)", "total_trainval": len(trainval), "total_test": len(testset),
        "n_train": N_TRAIN, "n_val": N_VAL, "n_test": N_TEST, "img_size": IMG,
        "fg_ratio_test": round(float(Yte.mean()), 4)}
print(DATA)

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def to_tensor(X): return ((torch.from_numpy(X).permute(0, 3, 1, 2).float() / 255.) - MEAN) / STD
Ttr, Tva, Tte = to_tensor(Xtr), to_tensor(Xva), to_tensor(Xte)
Mtr, Mva, Mte = [torch.from_numpy(y).long() for y in (Ytr, Yva, Yte)]

## 3. 평가 지표: IoU · Dice

In [ ]:
def iou_dice(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = (pred & gt).sum(); union = (pred | gt).sum(); s = pred.sum() + gt.sum()
    iou = inter / union if union else 1.0
    dice = 2 * inter / s if s else 1.0
    return float(iou), float(dice)

def summarize(preds, gts):
    v = np.array([iou_dice(p, g) for p, g in zip(preds, gts)])
    acc = float(np.mean([(p.astype(bool) == g.astype(bool)).mean() for p, g in zip(preds, gts)]))
    return {"IoU": round(float(v[:, 0].mean()), 4), "Dice": round(float(v[:, 1].mean()), 4),
            "IoU_std": round(float(v[:, 0].std()), 4), "PixelAcc": round(acc, 4)}, v[:, 0]

## 4. 전용 학습 모델: U-Net / SegFormer 파인튜닝

In [ ]:
def build(name):
    if name == "U-Net (ResNet34)":
        return smp.Unet("resnet34", encoder_weights="imagenet", in_channels=3, classes=2)
    if name == "SegFormer-B0":
        return SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/mit-b0", num_labels=2, id2label={0: "background", 1: "pet"}, label2id={"background": 0, "pet": 1})

def forward(model, x):
    out = model(pixel_values=x).logits if isinstance(model, transformers.PreTrainedModel) else model(x)
    if out.shape[-1] != x.shape[-1]:
        out = F.interpolate(out, size=x.shape[-2:], mode="bilinear", align_corners=False)
    return out   # (B, 2, H, W)

def dice_loss(logits, y):
    p = logits.softmax(1)[:, 1]; t = (y == 1).float()
    return 1 - (2 * (p * t).sum((1, 2)) + 1) / (p.sum((1, 2)) + t.sum((1, 2)) + 1)

@torch.no_grad()
def predict(model, T, bs=32):
    model.eval(); outs = []; times = []
    for i in range(0, len(T), bs):
        x = T[i:i + bs].to(device)
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        o = forward(model, x).argmax(1)
        if device == "cuda": torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) / len(x)); outs.append(o.cpu().numpy())
    return np.concatenate(outs), float(np.mean(times[1:] if len(times) > 1 else times)) * 1000

def train(name, lr):
    seed_all(); model = build(name).to(device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = EPOCHS * ((N_TRAIN + BATCH - 1) // BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=(device == "cuda"))
    hist = {"epoch": [], "train_loss": [], "val_loss": [], "val_IoU": [], "val_Dice": []}

    def evaluate():
        model.eval(); losses = []
        with torch.no_grad():
            for i in range(0, N_VAL, 32):
                x, y = Tva[i:i+32].to(device), Mva[i:i+32].to(device)
                o = forward(model, x); losses.append((F.cross_entropy(o, y) + dice_loss(o, y).mean()).item())
        p, _ = predict(model, Tva); m, _ = summarize(p, Yva)
        return float(np.mean(losses)), m

    vl, m = evaluate()      # 학습 전 (epoch 0)
    hist["epoch"].append(0); hist["train_loss"].append(None); hist["val_loss"].append(vl)
    hist["val_IoU"].append(m["IoU"]); hist["val_Dice"].append(m["Dice"])
    print(f"[{name}] epoch 0 (before) val_loss={vl:.3f} IoU={m['IoU']:.3f}")
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        model.train(); perm = torch.randperm(N_TRAIN); tl = []
        for i in range(0, N_TRAIN, BATCH):
            b = perm[i:i + BATCH]; x, y = Ttr[b].to(device), Mtr[b].to(device)
            if random.random() < 0.5: x, y = x.flip(-1), y.flip(-1)     # 좌우 반전 증강
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
                o = forward(model, x)
                loss = F.cross_entropy(o.float(), y) + dice_loss(o.float(), y).mean()
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl.append(loss.item())
        vl, m = evaluate()
        hist["epoch"].append(ep); hist["train_loss"].append(float(np.mean(tl))); hist["val_loss"].append(vl)
        hist["val_IoU"].append(m["IoU"]); hist["val_Dice"].append(m["Dice"])
        print(f"[{name}] epoch {ep} train_loss={np.mean(tl):.3f} val_loss={vl:.3f} IoU={m['IoU']:.3f} Dice={m['Dice']:.3f}")
    train_time = time.time() - t_start
    pred, ms = predict(model, Tte); met, per = summarize(pred, Yte); met_s, _ = summarize(pred, Yte_strict)
    info = {"params_M": round(n_params, 1), "lr": lr, "epochs": EPOCHS, "train_time_s": round(train_time, 1),
            "infer_ms_per_img": round(ms, 2), "before_val_IoU": hist["val_IoU"][0], "before_val_Dice": hist["val_Dice"][0]}
    print(name, info, met)
    del model; torch.cuda.empty_cache()
    return {"info": info, "test": met, "test_strict_gt": met_s, "hist": hist}, pred, per

RES, PRED, PER = {}, {}, {}
for name, lr in [("U-Net (ResNet34)", 3e-4), ("SegFormer-B0", 3e-4)]:
    RES[name], PRED[name], PER[name] = train(name, lr)

## 5. 파운데이션 모델: SAM zero-shot (박스 프롬프트 / 점 프롬프트)

In [ ]:
sam = SamModel.from_pretrained("facebook/sam-vit-base").to(device).eval()
sam_proc = SamProcessor.from_pretrained("facebook/sam-vit-base")
sam_params = sum(p.numel() for p in sam.parameters()) / 1e6

def sam_mask(pred_masks, orig_hw, scores=None):
    # pred_masks: (1, 1, K, 256, 256) 저해상도 logits -> 원본 크기 마스크 (직접 후처리)
    h, w = orig_hw; s = 1024 / max(h, w); rh, rw = int(h * s + 0.5), int(w * s + 0.5)
    m = pred_masks[0, 0]                               # (K, 256, 256)
    k = int(scores[0, 0].argmax()) if scores is not None else 0
    m = F.interpolate(m[k:k+1, None].float(), (1024, 1024), mode="bilinear", align_corners=False)[:, :, :rh, :rw]
    m = F.interpolate(m, (h, w), mode="bilinear", align_corners=False)[0, 0]
    return (m > 0).cpu().numpy()

def prompt_from_gt(gt):
    ys, xs = np.where(gt > 0)
    box = [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]
    dist = distance_transform_edt(np.pad(gt, 1))[1:-1, 1:-1]   # 이미지 테두리도 경계로 취급 → 물체 '한가운데' 클릭
    py, px = np.unravel_index(dist.argmax(), dist.shape)
    return box, [float(px), float(py)]

SAM_PRED = {"SAM (box prompt)": [], "SAM (point prompt)": []}
t_enc, t_dec = [], []
with torch.no_grad():
    for img, gt in test_orig:
        box, pt = prompt_from_gt(gt)
        inp = sam_proc(img, return_tensors="pt").to(device)
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter(); emb = sam.get_image_embeddings(inp["pixel_values"])
        if device == "cuda": torch.cuda.synchronize()
        t_enc.append(time.perf_counter() - t0)
        for key, kw, multi in [("SAM (box prompt)", {"input_boxes": [[box]]}, False),
                               ("SAM (point prompt)", {"input_points": [[[pt]]], "input_labels": [[[1]]]}, True)]:
            p = sam_proc(img, return_tensors="pt", **kw).to(device); p.pop("pixel_values", None)
            p = {k: (v.float() if v.is_floating_point() else v.long()) for k, v in p.items() if k in ("input_boxes", "input_points", "input_labels")}
            if device == "cuda": torch.cuda.synchronize()
            t0 = time.perf_counter()
            out = sam(image_embeddings=emb, multimask_output=multi, **p)
            if device == "cuda": torch.cuda.synchronize()
            t_dec.append(time.perf_counter() - t0)
            m = sam_mask(out.pred_masks, (img.height, img.width), out.iou_scores if multi else None)
            SAM_PRED[key].append(np.array(Image.fromarray(m.astype(np.uint8)).resize((IMG, IMG), Image.NEAREST)))

for key, preds in SAM_PRED.items():
    met, per = summarize(preds, Yte); met_s, _ = summarize(preds, Yte_strict)
    RES[key] = {"test_strict_gt": met_s, "info": {"params_M": round(sam_params, 1), "train_time_s": 0,
                         "encoder_ms_per_img": round(1000 * float(np.mean(t_enc[1:])), 1),
                         "decoder_ms_per_prompt": round(1000 * float(np.mean(t_dec[2:])), 1)}, "test": met}
    PRED[key], PER[key] = np.stack(preds), per
    print(key, RES[key])
del sam; torch.cuda.empty_cache()

## 6. 시각화

In [ ]:
names = list(RES.keys())
COL = {"U-Net (ResNet34)": "#2a9d8f", "SegFormer-B0": "#e76f51", "SAM (box prompt)": "#264653", "SAM (point prompt)": "#8d99ae"}

# (1) 학습 곡선
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for n in ["U-Net (ResNet34)", "SegFormer-B0"]:
    h = RES[n]["hist"]
    ax[0].plot(h["epoch"][1:], h["train_loss"][1:], "o-", color=COL[n], label=f"{n} train")
    ax[0].plot(h["epoch"], h["val_loss"], "s--", color=COL[n], alpha=.7, label=f"{n} val")
    ax[1].plot(h["epoch"], h["val_IoU"], "o-", color=COL[n], label=n)
ax[0].set_title("Loss (CE + Dice)"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
ax[1].set_title("Validation IoU"); ax[1].set_xlabel("epoch (0 = before fine-tuning)"); ax[1].set_ylim(0, 1); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig("results_w6/fig_curve.png", dpi=200); plt.show()

# (2) 테스트 IoU / Dice 비교
fig, ax = plt.subplots(figsize=(8, 4)); x = np.arange(len(names)); w = .38
b1 = ax.bar(x - w/2, [RES[n]["test"]["IoU"] for n in names], w, label="IoU")
b2 = ax.bar(x + w/2, [RES[n]["test"]["Dice"] for n in names], w, label="Dice")
ax.bar_label(b1, fmt="%.3f", fontsize=8); ax.bar_label(b2, fmt="%.3f", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=9); ax.set_ylim(0, 1.05); ax.legend(); ax.set_title(f"Test set ({N_TEST} images)")
plt.tight_layout(); plt.savefig("results_w6/fig_metrics.png", dpi=200); plt.show()

# (3) 이미지별 IoU 분포
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([PER[n] for n in names], showfliers=True); ax.set_xticks(range(1, len(names)+1)); ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel("per-image IoU"); ax.set_title("Per-image IoU distribution"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("results_w6/fig_box.png", dpi=200); plt.show()

# (4) 정성 비교: 원본 / GT / 각 모델 (+ SAM 프롬프트 표시)
def overlay(ax, img, mask, title, color=(0.1, 0.6, 1.0)):
    ax.imshow(img); m = np.zeros((*mask.shape, 4)); m[mask > 0] = (*color, .5); ax.imshow(m); ax.set_title(title, fontsize=8); ax.axis("off")
show = list(np.random.RandomState(1).choice(N_TEST, 3, replace=False)) + [int(np.argmin(PER["SAM (point prompt)"]))]
fig, axes = plt.subplots(len(show), 2 + len(names), figsize=(3 * (2 + len(names)), 3 * len(show)))
for r, i in enumerate(show):
    axes[r, 0].imshow(Xte[i]); axes[r, 0].set_title("Input" + (" (SAM-point worst)" if r == len(show)-1 else ""), fontsize=8); axes[r, 0].axis("off")
    overlay(axes[r, 1], Xte[i], Yte[i], "Ground truth", (0.2, 0.8, 0.2))
    for c, n in enumerate(names):
        overlay(axes[r, 2 + c], Xte[i], PRED[n][i], f"{n}\nIoU={PER[n][i]:.3f}")
    box, pt = prompt_from_gt(Yte[i])
    axes[r, 2 + names.index("SAM (box prompt)")].add_patch(plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], fill=False, color="yellow", lw=1.5))
    axes[r, 2 + names.index("SAM (point prompt)")].plot(pt[0], pt[1], "*", color="yellow", ms=12)
plt.tight_layout(); plt.savefig("results_w6/fig_qualitative.png", dpi=120); plt.show()

## 7. 결과 저장 및 다운로드

In [ ]:
out = {"env": ENV, "data": DATA, "seed": SEED,
       "models": {n: {k: v for k, v in RES[n].items()} for n in names}}
json.dump(out, open("results_w6/results.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps(out, ensure_ascii=False, indent=2))
with zipfile.ZipFile("results_w6.zip", "w") as z:
    for f in os.listdir("results_w6"): z.write(f"results_w6/{f}")
try:
    from google.colab import files; files.download("results_w6.zip")
except Exception as e:
    print("왼쪽 파일 탭에서 results_w6.zip을 직접 다운로드하세요:", e)